In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd
import numpy as np
import re

from src.data_processing import extract_game_info, extract_seed_value

In [58]:
m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
w_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

# Load submission data
submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")
submission_df[["Season", "TeamID1", "TeamID2"]] = (
    submission_df["ID"].apply(extract_game_info).apply(pd.Series)
)

In [9]:
# Define function to determine gender based on TeamID1
def determine_gender(team_id):
    if str(team_id).startswith("1"):  # Men's teams start with 1
        return "Men"
    elif str(team_id).startswith("3"):  # Women's teams start with 3
        return "Women"
    else:
        return None  # Handle unexpected cases


# Assign Gender column
submission_df["Gender"] = submission_df["TeamID1"].apply(determine_gender)

# Split into men's and women's submission dataframes
m_submission = submission_df[submission_df["Gender"] == "Men"].copy()
w_submission = submission_df[submission_df["Gender"] == "Women"].copy()


# Merge with seed_df to get Seed Values
def merge_with_seed(sub_df, seed_df):
    sub_df = sub_df.merge(
        seed_df,
        left_on=["Season", "TeamID1"],
        right_on=["Season", "TeamID"],
        how="inner",
    )
    sub_df = sub_df.rename(columns={"Seed": "SeedA"}).drop(columns=["TeamID"])

    sub_df = sub_df.merge(
        seed_df,
        left_on=["Season", "TeamID2"],
        right_on=["Season", "TeamID"],
        how="inner",
    )
    sub_df = sub_df.rename(columns={"Seed": "SeedB"}).drop(columns=["TeamID"])

    return sub_df


# Merge seed data for both men's and women's submissions
m_submission = merge_with_seed(m_submission, m_seed)
w_submission = merge_with_seed(w_submission, w_seed)

In [ ]:
m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
w_results = pd.read_csv(r"data\kaggle\WNCAATourneyCompactResults.csv")

In [34]:
# Function to merge results_df with seeds_df and tourney_round_lookup
def process_results(results_df, seeds_df, tourney_round_lookup):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    results_df["WSeed"] = results_df["Seed"]
    results_df["LSeed"] = results_df["Seed_T2"]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    results_df = results_df[
        [
            "Season",
            "DayNum",
            "WTeamID",
            "WSeed",
            "WScore",
            "LTeamID",
            "LSeed",
            "LScore",
            "WLoc",
            "NumOT",
            "Round",
            "Slot",
        ]
    ]

    return results_df

In [ ]:
m_results = process_results(m_results, m_seed, tourney_round_lookup)
w_results = process_results(w_results, w_seed, tourney_round_lookup)

In [35]:
m_results.head()

,Season,DayNum,WTeamID,WSeed,WScore,LTeamID,LSeed,LScore,WLoc,NumOT,Round,Slot
0,1985,136,1116,X09,63,1234,X08,54,N,0,1.0,R1X8
1,1985,136,1120,Z11,59,1345,Z06,58,N,0,1.0,R1Z6
2,1985,136,1207,W01,68,1250,W16,43,N,0,1.0,R1W1
3,1985,136,1229,Y09,58,1425,Y08,55,N,0,1.0,R1Y8
4,1985,136,1242,Z03,49,1325,Z14,38,N,0,1.0,R1Z3


In [36]:
w_results.head()

,Season,DayNum,WTeamID,WSeed,WScore,LTeamID,LSeed,LScore,WLoc,NumOT,Round,Slot
0,1998,137,3104,X02,94,3422,X15,46,H,0,1.0,R1X2
1,1998,137,3112,W03,75,3365,W14,63,H,0,1.0,R1W3
2,1998,137,3163,W02,93,3193,W15,52,H,0,1.0,R1W2
3,1998,137,3198,Y07,59,3266,Y10,45,H,0,1.0,R1Y7
4,1998,137,3203,W10,74,3208,W07,72,A,0,1.0,R1W7


In [76]:
# Load submission data
submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")
submission_df[["Season", "TeamID1", "TeamID2"]] = (
    submission_df["ID"].apply(extract_game_info).apply(pd.Series)
)

# Assign Gender column
submission_df["Gender"] = submission_df["TeamID1"].apply(determine_gender)

# Split into men's and women's submission dataframes
m_submission = submission_df[submission_df["Gender"] == "Men"].copy()
w_submission = submission_df[submission_df["Gender"] == "Women"].copy()

In [77]:
# Function to merge sub_df with seeds_df and tourney_round_lookup
def process_submission(sub_df, seeds_df, tourney_round_lookup):
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID1"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID2"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    sub_df["Seed1"] = sub_df["Seed"].fillna("")
    sub_df["Seed2"] = sub_df["Seed_T2"].fillna("")

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    sub_df["StrongSeed"] = sub_df[["Seed1", "Seed2"]].min(axis=1)
    sub_df["WeakSeed"] = sub_df[["Seed1", "Seed2"]].max(axis=1)

    sub_df = sub_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    sub_df = sub_df[
        [
            "ID",
            "Pred",
            "Season",
            "TeamID1",
            "Seed1",
            "TeamID2",
            "Seed2",
            "Round",
            "Slot",
        ]
    ]

    return sub_df

In [78]:
m_submission = process_submission(m_submission, m_seed, tourney_round_lookup)
w_submission = process_submission(w_submission, w_seed, tourney_round_lookup)

In [79]:
m_submission.head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed1,Round,Slot
0,2021_1101_1102,0.5,2021,1101,W14,1102,W14,NaN,NaN
1,2021_1101_1103,0.5,2021,1101,W14,1103,W14,NaN,NaN
2,2021_1101_1104,0.5,2021,1101,W14,1104,W14,3.0,R3W2
3,2021_1101_1105,0.5,2021,1101,W14,1105,W14,NaN,NaN
4,2021_1101_1106,0.5,2021,1101,W14,1106,W14,NaN,NaN


In [82]:
w_submission.head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed1,Round,Slot
0,2021_3101_3102,0.5,2021,3101,,3102,,NaN,NaN
1,2021_3101_3103,0.5,2021,3101,,3103,,NaN,NaN
2,2021_3101_3104,0.5,2021,3101,,3104,,NaN,NaN
3,2021_3101_3105,0.5,2021,3101,,3105,,NaN,NaN
4,2021_3101_3106,0.5,2021,3101,,3106,,NaN,NaN
